# Demo Pipeline: рекомендательная система маршрутов

Этот ноутбук показывает полный MVP-пайплайн рекомендательной системы для каталога hiking/tourism маршрутов на полностью синтетических данных.

В проекте нет клиентских данных, production-кода, proprietary database schema, внутренних бизнес-метрик или customer-specific business logic.

## 1. Загрузка synthetic dataset

Стартовая точка пайплайна — воспроизводимые CSV-файлы в `data/`. Первая ячейка также добавляет локальный `src/` в `sys.path`, чтобы ноутбук запускался из Jupyter без обязательного `pip install -e .`.

In [10]:
from pathlib import Path
import sys

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from hiking_recommender.data_loader import load_demo_dataset

dataset = load_demo_dataset(project_root / "data")

{
    "users": len(dataset.users),
    "routes": len(dataset.routes),
    "train_interactions": len(dataset.train_interactions),
    "test_interactions": len(dataset.test_interactions),
}

{'users': 200,
 'routes': 120,
 'train_interactions': 4174,
 'test_interactions': 1042}

**Резюме этапа:** загружены `200` synthetic users, `120` synthetic routes, `4174` train interactions и `1042` test interactions. Этого достаточно для компактной demo-проверки warm-user, cold-start/fallback и offline evaluation без исследовательской сложности.

In [11]:
dataset.routes.head()

,duration_hours,route_id,route_tags,length_km,popularity,season,region,elevation_gain_m,difficulty
0,1.0,route_001,family|forest|waterfall,2.5,0.480,winter,east,210,easy
1,1.5,route_002,forest|wildlife,4.8,0.186,summer,central,99,easy
2,4.7,route_003,family|forest|lake,12.2,0.731,winter,central,279,moderate
3,1.6,route_004,waterfall|wildlife,4.5,0.686,autumn,north,100,easy
4,4.2,route_005,forest|viewpoint|waterfall,7.9,0.263,summer,north,774,moderate


**Резюме по данным маршрутов:** route catalog содержит только generic synthetic признаки: `route_id`, `region`, `length_km`, `duration_hours`, `elevation_gain_m`, `difficulty`, `season`, `route_tags`, `popularity`. В примере нет реальных названий маршрутов или регионов; используются публичные synthetic IDs вида `route_001`.

## 2. Feature engineering

Feature layer отделён от загрузки данных. Здесь строятся route-level признаки и user-item matrix по implicit feedback weights.

In [12]:
from hiking_recommender.features import build_features, build_seen_routes

features = build_features(dataset)
route_features = features["route_features"]
user_item_matrix = features["user_item_matrix"]

{
    "route_features_shape": route_features.shape,
    "user_item_matrix_shape": user_item_matrix.shape,
}

{'route_features_shape': (120, 19), 'user_item_matrix_shape': (200, 120)}

**Резюме этапа:** получена feature matrix маршрутов размера `(120, 19)` и user-item matrix размера `(200, 120)`. Это подтверждает, что все users/routes представлены в train контуре, а retrieval-модели могут работать поверх общей матрицы взаимодействий.

## 3. Retrieval sources: запуск моделей по отдельности

Сначала запускаем три модели независимо друг от друга. Это важно для baseline-first подхода: до hybrid merge нужно понимать, что даёт каждый retrieval source сам по себе.

Источники кандидатов:

- `PopularityRecommender` — популярные маршруты и fallback;
- `CollaborativeRecommender` — item-item similarity по implicit feedback;
- `ContentBasedRecommender` — похожесть маршрутов по признакам.

In [13]:
from hiking_recommender.baseline import PopularityRecommender
from hiking_recommender.collaborative import CollaborativeRecommender
from hiking_recommender.content_based import ContentBasedRecommender

popularity = PopularityRecommender().fit(dataset)
collaborative = CollaborativeRecommender().fit(dataset)
content = ContentBasedRecommender().fit(dataset)

user_id = "user_001"
region = "north"
candidate_limit = 30

popularity_candidates = popularity.recommend(user_id, top_k=candidate_limit, region=region)
collaborative_candidates = collaborative.recommend(user_id, top_k=candidate_limit, region=region)
content_candidates = content.recommend(user_id, top_k=candidate_limit, region=region)

{
    "popularity": len(popularity_candidates),
    "collaborative": len(collaborative_candidates),
    "content": len(content_candidates),
}

{'popularity': 18, 'collaborative': 18, 'content': 18}

**Резюме этапа:** для `user_001` и региона `north` каждый retrieval source вернул по `18` кандидатов. Это значит, что region filter не обнуляет выдачу, а все три источника можно корректно сравнивать и затем объединять.

## 4. Сравнение отдельных моделей

Считаем offline top-K metrics отдельно для popularity, collaborative и content моделей. Это показывает, какой источник сильнее до merger и business rules.

In [14]:
import pandas as pd

from hiking_recommender.evaluation import build_relevance, evaluate_recommendations

cutoff = 10
user_ids = sorted(build_relevance(dataset.test_interactions))
all_route_ids = list(dataset.routes["route_id"].astype(str))


def route_ids(candidates):
    return [candidate.route_id for candidate in candidates]


single_model_recommendations = {
    "popularity": {
        current_user_id: route_ids(popularity.recommend(current_user_id, top_k=cutoff))
        for current_user_id in user_ids
    },
    "collaborative": {
        current_user_id: route_ids(collaborative.recommend(current_user_id, top_k=cutoff))
        for current_user_id in user_ids
    },
    "content": {
        current_user_id: route_ids(content.recommend(current_user_id, top_k=cutoff))
        for current_user_id in user_ids
    },
}

single_model_metrics = pd.DataFrame(
    [
        {"model": model_name, **evaluate_recommendations(recommendations, dataset.test_interactions, all_route_ids, cutoff=cutoff)}
        for model_name, recommendations in single_model_recommendations.items()
    ]
).sort_values("model")

metric_columns = [column for column in single_model_metrics.columns if column != "model"]
single_model_metrics_table = single_model_metrics.copy()
single_model_metrics_table[metric_columns] = single_model_metrics_table[metric_columns].round(4)

single_model_metrics_table

,model,precision@10,recall@10,map@10,ndcg@10,coverage@10
1,collaborative,0.0790,0.1484,0.0572,0.1249,0.9167
2,content,0.1035,0.1898,0.0740,0.1584,1.0000
0,popularity,0.0735,0.1375,0.0498,0.1130,0.1833


**Анализ отдельных моделей:**

- `popularity` — самый простой и самый стабильный fallback, но coverage низкий (`coverage@10=0.1833`): модель часто рекомендует ограниченный набор популярных маршрутов.
- `collaborative` улучшает ranking относительно popularity (`ndcg@10=0.1249` против `0.1130`) и резко расширяет coverage (`0.9167`), потому что использует поведенческие связи route-route.
- `content` — лучший отдельный источник в этом прогоне: `precision@10=0.1035`, `recall@10=0.1898`, `ndcg@10=0.1584`, `coverage@10=1.0000`. На synthetic данных признаки маршрутов хорошо совпадают с пользовательскими предпочтениями.

## 5. Candidate merger: hybrid retrieval

Теперь объединяем три источника. Merger работает по `route_id`, удаляет дубликаты и суммирует weighted reciprocal-rank contribution. Кандидаты, найденные несколькими источниками, получают boost.

In [15]:
from hiking_recommender.merger import merge_candidates

merged = merge_candidates(
    [collaborative_candidates, content_candidates, popularity_candidates],
    top_k=candidate_limit,
)

pd.DataFrame(
    [
        {
            "route_id": candidate.route_id,
            "rank": candidate.final_rank,
            "score": round(candidate.merged_score, 5),
            "sources": ",".join(candidate.sources),
            "n_sources": candidate.n_sources,
        }
        for candidate in merged[:10]
    ]
)

,route_id,rank,score,sources,n_sources
0,route_095,1,0.03770,"collaborative,content,popularity",3
1,route_109,2,0.03595,"collaborative,content,popularity",3
2,route_102,3,0.03590,"collaborative,content,popularity",3
3,route_080,4,0.03536,"collaborative,content,popularity",3
4,route_094,5,0.03508,"collaborative,content,popularity",3
5,route_054,6,0.03394,"collaborative,content,popularity",3
6,route_010,7,0.03362,"collaborative,content,popularity",3
7,route_030,8,0.03338,"collaborative,content,popularity",3
8,route_068,9,0.03303,"collaborative,content,popularity",3
9,route_075,10,0.03296,"collaborative,content,popularity",3


**Анализ merger на примере пользователя:** top-10 после merge состоит из маршрутов, найденных всеми тремя источниками: `collaborative`, `content`, `popularity`. Это демонстрирует ожидаемый hybrid-паттерн: пересечение независимых сигналов повышает confidence, а итоговый список остаётся deduplicated.

In [16]:
hybrid_recommendations = {
    current_user_id: route_ids(
        merge_candidates(
            [
                collaborative.recommend(current_user_id, top_k=candidate_limit),
                content.recommend(current_user_id, top_k=candidate_limit),
                popularity.recommend(current_user_id, top_k=candidate_limit),
            ],
            top_k=cutoff,
        )
    )
    for current_user_id in user_ids
}

hybrid_metrics = pd.DataFrame(
    [
        {"model": "hybrid", **evaluate_recommendations(hybrid_recommendations, dataset.test_interactions, all_route_ids, cutoff=cutoff)}
    ]
)

pd.concat([single_model_metrics, hybrid_metrics], ignore_index=True).sort_values("model")

,model,precision@10,recall@10,map@10,ndcg@10,coverage@10
0,collaborative,0.0790,0.148393,0.057212,0.124890,0.916667
1,content,0.1035,0.189756,0.073973,0.158405,1.000000
3,hybrid,0.1055,0.193173,0.074078,0.160452,0.950000
2,popularity,0.0735,0.137536,0.049824,0.113029,0.183333


**Изменение метрик после merger:** hybrid даёт лучший результат среди вариантов до business rules: `precision@10=0.1055`, `recall@10=0.1932`, `ndcg@10=0.1605`. Прирост относительно лучшей отдельной модели (`content`) небольшой, но ожидаемый для MVP: merger усиливает кандидатов, которые подтверждаются несколькими источниками. Coverage снижается с `1.0000` у content до `0.9500`, потому что hybrid чаще выбирает согласованные кандидаты, а не максимально широкий catalog spread.

## 6. Business rules

Business rules применяются после retrieval и merge. В этом demo используются hard filters по региону, максимальной сложности, исключение уже виденных маршрутов и fallback fill. Fallback не должен обходить hard filters.

In [17]:
from hiking_recommender.business_rules import apply_business_rules

seen_routes = build_seen_routes(dataset.train_interactions)
fallback = merge_candidates([popularity_candidates], top_k=candidate_limit)

final_recommendations = apply_business_rules(
    merged,
    routes=dataset.routes,
    top_k=10,
    region=region,
    max_difficulty="moderate",
    seen_route_ids=seen_routes.get(user_id, set()),
    fallback_candidates=fallback,
)

route_lookup = dataset.routes.set_index("route_id")
pd.DataFrame(
    [
        {
            "route_id": candidate.route_id,
            "rank": candidate.final_rank,
            "score": round(candidate.merged_score, 5),
            "region": route_lookup.loc[candidate.route_id, "region"],
            "difficulty": route_lookup.loc[candidate.route_id, "difficulty"],
            "sources": ",".join(candidate.sources),
        }
        for candidate in final_recommendations
    ]
)

,route_id,rank,score,region,difficulty,sources
0,route_095,1,0.03770,north,easy,"collaborative,content,popularity"
1,route_109,2,0.03595,north,easy,"collaborative,content,popularity"
2,route_102,3,0.03590,north,easy,"collaborative,content,popularity"
3,route_080,4,0.03536,north,moderate,"collaborative,content,popularity"
4,route_094,5,0.03508,north,easy,"collaborative,content,popularity"
5,route_054,6,0.03394,north,moderate,"collaborative,content,popularity"
6,route_010,7,0.03362,north,moderate,"collaborative,content,popularity"
7,route_030,8,0.03338,north,moderate,"collaborative,content,popularity"
8,route_075,9,0.03296,north,moderate,"collaborative,content,popularity"
9,route_018,10,0.03291,north,easy,"collaborative,content,popularity"


**Анализ business rules на примере пользователя:** финальная выдача содержит `10` маршрутов только из региона `north`; сложности — `easy` и `moderate`, то есть фильтр `max_difficulty='moderate'` соблюдён. Ранги пересчитаны в диапазоне `1..10`, дубликатов `route_id` нет.

In [18]:
hybrid_with_rules_recommendations = {
    current_user_id: route_ids(
        apply_business_rules(
            merge_candidates(
                [
                    collaborative.recommend(current_user_id, top_k=candidate_limit),
                    content.recommend(current_user_id, top_k=candidate_limit),
                    popularity.recommend(current_user_id, top_k=candidate_limit),
                ],
                top_k=candidate_limit,
            ),
            routes=dataset.routes,
            top_k=cutoff,
            max_difficulty="moderate",
            seen_route_ids=seen_routes.get(current_user_id, set()),
            fallback_candidates=merge_candidates(
                [popularity.recommend(current_user_id, top_k=candidate_limit)],
                top_k=candidate_limit,
            ),
        )
    )
    for current_user_id in user_ids
}

hybrid_with_rules_metrics = pd.DataFrame(
    [
        {"model": "hybrid_with_rules", **evaluate_recommendations(hybrid_with_rules_recommendations, dataset.test_interactions, all_route_ids, cutoff=cutoff)}
    ]
)

full_metrics = pd.concat(
    [single_model_metrics, hybrid_metrics, hybrid_with_rules_metrics],
    ignore_index=True,
).sort_values("model")

full_metrics

,model,precision@10,recall@10,map@10,ndcg@10,coverage@10
0,collaborative,0.0790,0.148393,0.057212,0.124890,0.916667
1,content,0.1035,0.189756,0.073973,0.158405,1.000000
3,hybrid,0.1055,0.193173,0.074078,0.160452,0.950000
4,hybrid_with_rules,0.1000,0.185238,0.074095,0.156739,0.800000
2,popularity,0.0735,0.137536,0.049824,0.113029,0.183333


**Изменение метрик после `apply_business_rules`**:

после правил ranking metrics немного снижаются:
- `precision@10` с `0.1055` до `0.1000`,
- `recall@10` с `0.1932` до `0.1852`,
- `ndcg@10` с `0.1605` до `0.1567`.
- Coverage снижается с `0.9500` до `0.8000`.

- Это нормальный trade-off: hard filters ограничивают пространство рекомендаций, зато выдача становится продуктово безопаснее и соответствует ограничениям пользователя (`max_difficulty`, exclude seen, fallback без обхода фильтров).

## 7. API payload example

Тот же pipeline доступен через FastAPI endpoint `POST /recommendations`. Ноутбук не поднимает сервер, а показывает payload, который соответствует публичному API contract.

In [19]:
api_payload = {
    "user_id": user_id,
    "region": region,
    "top_k": 5,
    "max_difficulty": "moderate",
}

api_payload

{'user_id': 'user_001',
 'region': 'north',
 'top_k': 5,
 'max_difficulty': 'moderate'}

**Резюме этапа:** API-запрос минимальный и продуктово понятный: `user_id`, optional `region`, `top_k`, optional `max_difficulty`. Такой контракт удобно показывать в README, curl-примере и smoke-тестах API.